# A — Agent LangChain

Vidéos du cours. Kernel **ap5_agentique**. Ollama `qwen2.5:3b`.


## 1 — Agent simple


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from langchain.agents import create_agent
from langchain_core.tools import tool
from ap5.llm import modele_ollama

llm = modele_ollama()
print(llm.invoke("Dis bonjour en une phrase.").content)


## 2 — Mémoire

Checkpointer + `thread_id` : l’agent se souvient du fil.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memoire = InMemorySaver()
agent_mem = create_agent(llm, tools=[], checkpointer=memoire, system_prompt="Réponds bref.")
cfg = {"configurable": {"thread_id": "seance-1"}}
print(agent_mem.invoke({"messages": [("human", "Je m'appelle Imane.")]}, cfg)["messages"][-1].content)
print(agent_mem.invoke({"messages": [("human", "Quel est mon prénom ?")]}, cfg)["messages"][-1].content)


## 3 — Tools maison


In [ ]:
@tool
def heure_paris() -> str:
    """Heure locale approximative du poste."""
    from datetime import datetime
    return datetime.now().isoformat(timespec="seconds")

agent_tools = create_agent(llm, tools=[heure_paris], system_prompt="Utilise le tool si on demande l'heure.")
print(agent_tools.invoke({"messages": [("human", "Quelle heure est-il ?")]})["messages"][-1].content)


## 4 — Tools prédéfinis : DuckDuckGo, Tavily, PythonREPL


In [ ]:
import os
from langchain_core.tools import tool
from langchain_experimental.tools import PythonREPLTool

@tool
def duckduckgo_search(query: str) -> str:
    """Recherche web DuckDuckGo."""
    from ddgs import DDGS
    with DDGS() as ddg:
        hits = list(ddg.text(query, max_results=3))
    return "\n".join(str(h) for h in hits) or "Aucun résultat."

outils = [duckduckgo_search, PythonREPLTool()]
if os.getenv("TAVILY_API_KEY"):
    from langchain_community.tools.tavily_search import TavilySearchResults
    outils.append(TavilySearchResults(max_results=3))
    print("Tavily activé")
else:
    print("Tavily : définir TAVILY_API_KEY dans .env")

agent_web = create_agent(llm, tools=outils, system_prompt="Tools web / Python si besoin. Réponds en français.")
print(agent_web.invoke({"messages": [("human", "Cherche ce qu'est LangGraph en 2 phrases.")]})["messages"][-1].content)


## 5 — Middlewares

`dynamic_prompt`, `dynamic_model` (`wrap_model_call`), `ToolErrorMiddleware`, GuardRails (`PIIMiddleware`), Human in the loop sur le REPL.


In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from ap5.middlewares import liste_middlewares

agent_mw = create_agent(
    llm,
    tools=outils,
    middleware=liste_middlewares(hitl=False),
    checkpointer=InMemorySaver(),
    system_prompt="Assistant de TP.",
)
cfg2 = {"configurable": {"thread_id": "mw-1"}}
print(agent_mw.invoke({"messages": [("human", "Mon email est  test@ecole.fr , redis-le.")]}, cfg2)["messages"][-1].content)


HITL : interrompt avant d’exécuter `Python_REPL` (décommenter pour tester).


In [ ]:
# agent_hitl = create_agent(
#     llm, tools=[PythonREPLTool()],
#     middleware=[HumanInTheLoopMiddleware(interrupt_on={"Python_REPL": True})],
#     checkpointer=InMemorySaver(),
# )
# etat = agent_hitl.invoke({"messages": [("human", "Calcule 2**10 avec Python")]}, {"configurable": {"thread_id": "hitl"}})
# print(etat)
# # reprendre : agent_hitl.invoke(Command(resume={"decisions": [{"type": "approve"}]}), {"configurable": {"thread_id": "hitl"}})
print("HITL configuré — décommenter la cellule pour un run interactif.")


## 6 — Chatbot RAG agentique


In [ ]:
from ap5.agentic_rag import construire_graphe_rag
from langchain_core.messages import HumanMessage

rag = construire_graphe_rag()
q = "Que signifie le voyant orange clignotant de la PX-200 ?"
out = rag.invoke({"messages": [HumanMessage(content=q)]})
print(out["messages"][-1].content)
